# Intake: Reizman_Summit (Suzuki–Miyaura in flow, continuous conditions)

**Dataset `reizman_summit`** — named for the data's origin: Reizman *et al.*'s
measurements, as redistributed by the Summit package. Fu *et al.* is the paper
that modelled them and motivated adding them here.

Fu *et al.*, *Org. Chem. Front.* 2020, **7**, 2269 ([10.1039/d0qo00544d](https://doi.org/10.1039/d0qo00544d))
trained a DNN on Jensen's automated microfluidic Suzuki–Miyaura screen — the data
of Reizman *et al.*, *React. Chem. Eng.* 2016, **1**, 658
([10.1039/C6RE00153J](https://doi.org/10.1039/C6RE00153J)). Four substrate
pairs ("cases"), eight Buchwald-type precatalysts, and three conditions varied
**continuously**: temperature (30–110 °C), catalyst loading (0.5–2.5 mol%) and
residence time (1–10 min). Target: yield (%).

**Where the data comes from.** Fu's ESI is not scriptably downloadable, but the
reaction table it was built from is redistributed (MIT licence) by the Summit
benchmark package as `reizman_suzuki_case_{1..4}.csv`; this notebook downloads
those, pinned to one commit. They carry the reactions only — catalyst,
conditions, yield, TON — **not** Fu's 41 QM descriptors, which exist only in
their ESI. This bundle is therefore the *base* one: our feature sections only.
A Fu-descriptor bundle (their features + ours, ours in place of theirs) can be
added beside it, as `ahneman_doyle_dft` was for Ahneman, once the ESI table is
available.

Decisions made here, each explained where it is made:

* **Group = ligand** (not precatalyst). P1-L1 and P2-L1 share XPhos, so the
  scaffold (P1 = OMs, P2 = Cl) becomes a condition field and LOLO is genuinely
  leave-one-*ligand*-out.
* **XantPhos (L4) rows are dropped.** It is bidentate and absent from Kraken, so
  no Kraken section can describe it honestly; 53 of 385 rows.
* **Continuous conditions stay numeric** (`data.common_numeric`) in every
  section, standardised per fold — one-hot encoding a temperature would discard
  its order.

In [1]:
from datetime import date
import hashlib
import urllib.request
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()            # datasets/reizman_summit/
ROOT = HERE.parent.parent              # gp_collab_hub/
sys.path.insert(0, str(ROOT))

from gpc import bundle as bundle_mod
from gpc import prep
from gpc.config import (check_run_against_bundle, read_json, validate_config,
                        write_json, write_yaml)
from gpc.data import load_bundle
from gpc.features import _preprocessor, feature_frame, model_columns

pd.set_option("display.width", 220, "display.max_columns", 60, "display.max_colwidth", 70)

# ---- Editable -----------------------------------------------------------
DATASET      = "reizman_summit"
DISPLAY_NAME = "Reizman_Summit"
TARGET       = "yield"
GROUP        = "ligand"
CATEGORICAL  = ["aryl_halide", "boron_reagent", "scaffold"]       # one-hot conditions
NUMERIC      = ["temperature", "catalyst_loading", "t_res"]        # continuous conditions
SOURCE_URL   = "https://doi.org/10.1039/d0qo00544d"

SUMMIT_COMMIT = "249751d57c3603fda99437b35656a1d78e2f0371"   # last change to the case CSVs
RAW_URL = ("https://raw.githubusercontent.com/sustainable-processes/summit/"
           f"{SUMMIT_COMMIT}/summit/benchmarks/data/reizman_suzuki_case_{{case}}.csv")

# Substrates, with Fu's labels (their Fig. 1 / Scheme 1).
ARYL_HALIDES = {"1a": ("3-chloropyridine", "Clc1cccnc1"),
                "1b": ("2-chloropyridine", "Clc1ccccn1"),
                "1c": ("3-bromoquinoline", "Brc1cnc2ccccc2c1")}
BORON = {"2a": ("3,5-dimethylisoxazole-4-boronic acid pinacol ester",
                "Cc1noc(C)c1B1OC(C)(C)C(C)(C)O1"),
         "2b": ("benzofuran-2-boronic acid", "OB(O)c1cc2ccccc2o1"),
         "2c": ("N-Boc-pyrrole-2-boronic acid", "CC(C)(C)OC(=O)n1cccc1B(O)O")}
# Summit case -> Fu scheme; established from the data in section 2.
CASE_SCHEME = {1: ("1c", "2a"), 2: ("1a", "2a"), 3: ("1a", "2b"), 4: ("1b", "2c")}

# Precatalyst codes -> (scaffold, ligand). P1: X = OMs, P2: X = Cl (Fu Fig. 1).
LIGANDS = {"L1": "XPhos", "L2": "SPhos", "L3": "RuPhos", "L4": "XantPhos",
           "L5": "PCy3", "L6": "PPh3", "L7": "PtBu3"}
MAPPING = {"XPhos": 1, "SPhos": 3, "RuPhos": 4, "PCy3": 11, "PPh3": 17, "PtBu3": 8}
DROP_LIGANDS = ["XantPhos"]            # bidentate; not in Kraken

RAW = HERE / "raw"                     # download cache (git-ignored)
REFRESH = False
OUT = HERE / "inputs"
# --------------------------------------------------------------------------
print("hub:", ROOT)

hub: C:\Users\kagoble\Documents\GitHub\DOPE-MURI\gp_collab_hub


## 1. Download the four Reizman cases

From the Summit repository at a fixed commit, so the bytes — and their hashes,
recorded in `audit.json` — cannot drift.

In [2]:
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

RAW.mkdir(parents=True, exist_ok=True)
sources = {}
for case in CASE_SCHEME:
    url = RAW_URL.format(case=case)
    path = RAW / f"reizman_suzuki_case_{case}.csv"
    if REFRESH or not path.exists():
        with urllib.request.urlopen(url, timeout=120) as response:
            path.write_bytes(response.read())
        print("downloaded", path.name)
    sources[path.name] = {"url": url, "sha256": sha256(path), "bytes": path.stat().st_size}
display(pd.DataFrame(sources).T)

,url,sha256,bytes
reizman_suzuki_case_1.csv,https://raw.githubusercontent.com/sustainable-processes/summit/249...,346d976b89241662965f047b6725a9972941afb4c12105e82947cf9e2a27a450,3275
reizman_suzuki_case_2.csv,https://raw.githubusercontent.com/sustainable-processes/summit/249...,20f648b7e48a40059d4ba990161d0ad21cf255c45cdc90e6fa52a0e9b4150f3f,3216
reizman_suzuki_case_3.csv,https://raw.githubusercontent.com/sustainable-processes/summit/249...,cb57a2a8a4075ff312508e48673b3b4f22667b8fc609f7c60b57bb4f357a36b9,3309
reizman_suzuki_case_4.csv,https://raw.githubusercontent.com/sustainable-processes/summit/249...,fb2f2bd7eae04495f75a91996e309cc3f4f688a4fa02d7ef9e556246ce8bbf75,3446


## 2. What is in them, and which case is which scheme

Each file has a two-line header (Summit's `NAME` / `TYPE` rows), then one row
per experiment: `catalyst` (P*x*-L*y*), `t_res` (s), `temperature` (°C),
`catalyst_loading` (mol%), `ton` and `yld` (%). Nothing in the files names the
substrates, so the case → scheme assignment is made from the paper: Fu's
Table 1 reports, for each of its four modelled schemes, the optimum the
microfluidic platform found. Each case contains a measured row at exactly one
of those optima (and the low-yielding case is the scheme Fu flags as poor).

In [3]:
frames = []
for case in CASE_SCHEME:
    frame = pd.read_csv(RAW / f"reizman_suzuki_case_{case}.csv", skiprows=[1])
    frame.insert(0, "case", case)
    frames.append(frame)
raw = pd.concat(frames, ignore_index=True)
print(f"{len(raw)} experiments ({', '.join(f'case {c}: {n}' for c, n in raw.case.value_counts().sort_index().items())})")
print("Fu et al. report 387; Summit's tables hold 385.")
display(raw.groupby("case")[["t_res", "temperature", "catalyst_loading", "yld", "ton"]]
        .agg(["min", "max"]).round(2))
display(pd.crosstab(raw.catalyst, raw.case, margins=True))

# Fu Table 1, column a: the microfluidic optimum per scheme.
FU_OPTIMA = {("1c", "2a"): ("P1-L4", 110, 1.2), ("1a", "2a"): ("P1-L5", 110, 2.1),
             ("1a", "2b"): ("P1-L1", 110, 1.2), ("1b", "2c"): ("P1-L1", 97, 1.0)}
rows = []
for case, scheme in CASE_SCHEME.items():
    catalyst, temp, loading = FU_OPTIMA[scheme]
    part = raw[(raw.case == case) & (raw.catalyst == catalyst)]
    distance = ((part.temperature - temp) / 80) ** 2 + ((part.catalyst_loading - loading) / 2) ** 2
    near = part.loc[distance.idxmin()]
    best = raw[raw.case == case].sort_values("yld").iloc[-1]
    rows.append({"case": case, "scheme": "–".join(scheme), "Fu optimum": f"{catalyst} {temp} °C {loading} mol%",
                 "nearest measured": f"{near.temperature:.0f} °C {near.catalyst_loading:.2f} mol%, "
                                     f"{near.t_res:.0f} s -> {near.yld:.1f}%",
                 "case max yield": f"{best.yld:.1f}% ({best.catalyst})"})
display(pd.DataFrame(rows))

385 experiments (case 1: 96, case 2: 96, case 3: 96, case 4: 97)
Fu et al. report 387; Summit's tables hold 385.


t_res        temperature        catalyst_loading        yld        ton       
       min    max         min    max              min   max  min   max  min    max
case                                                                              
1     60.0  600.0        30.0  110.0             0.50  2.52  0.6  98.7  0.2   84.7
2     60.0  600.0        30.0  110.0             0.49  2.52  0.1  47.6  0.0   20.3
3     60.0  600.0        30.0  110.0             0.50  2.52  0.2  94.1  0.1   85.6
4     60.0  600.0        30.0  110.0             0.49  2.51  0.1  99.8  0.0  129.4

case,1,2,3,4,All
catalyst,,,,,
P1-L1,12,23,21,25,81
P1-L2,5,10,21,24,60
P1-L3,11,15,24,16,66
P1-L4,37,5,5,6,53
P1-L5,8,24,6,7,45
P1-L6,5,5,4,5,19
P1-L7,11,9,7,5,32
P2-L1,7,5,8,9,29
All,96,96,96,97,385


,case,scheme,Fu optimum,nearest measured,case max yield
0,1,1c–2a,P1-L4 110 °C 1.2 mol%,"110 °C 1.19 mol%, 600 s -> 72.1%",98.7% (P1-L4)
1,2,1a–2a,P1-L5 110 °C 2.1 mol%,"110 °C 2.01 mol%, 185 s -> 18.6%",47.6% (P1-L5)
2,3,1a–2b,P1-L1 110 °C 1.2 mol%,"110 °C 1.22 mol%, 600 s -> 90.6%",94.1% (P1-L2)
3,4,1b–2c,P1-L1 97 °C 1.0 mol%,"89 °C 0.98 mol%, 600 s -> 85.6%",99.8% (P1-L1)


## 3. Reshape into the hub schema

* `catalyst` splits into `scaffold` (P1/P2) and `ligand` (L1–L7 → name).
* `case` becomes `aryl_halide` + `boron_reagent` (names; SMILES in the audit).
* `yld` → `yield`; `ton` and `case` are kept for reference and ignored by the
  models.
* **XantPhos rows are dropped** (bidentate, not in Kraken). The audit records
  how many from each case.
* Repeated condition tuples are **kept**: they are replicate measurements. As
  with Gesmundo, a random fold can put one replicate in train and its twin in
  test; LOLO cannot, since a ligand moves as a whole.

In [4]:
codes = raw.catalyst.str.extract(r"^(P\d)-(L\d)$")
assert codes.notna().all().all(), "unexpected catalyst code"
table = pd.DataFrame({
    "case": raw.case,
    "catalyst": raw.catalyst,
    "scaffold": codes[0],
    "ligand": codes[1].map(LIGANDS),
    "aryl_halide": raw.case.map(lambda c: ARYL_HALIDES[CASE_SCHEME[c][0]][0]),
    "boron_reagent": raw.case.map(lambda c: BORON[CASE_SCHEME[c][1]][0]),
    "temperature": raw.temperature,
    "catalyst_loading": raw.catalyst_loading,
    "t_res": raw.t_res,
    TARGET: raw.yld,
    "ton": raw.ton,
})
assert table.notna().all().all()

dropped = table[table.ligand.isin(DROP_LIGANDS)]
reactions = table[~table.ligand.isin(DROP_LIGANDS)]
reactions = prep.attach_kraken(reactions, GROUP, MAPPING)
reactions = prep.add_row_ids(reactions)

keys = ["case", "catalyst", "temperature", "catalyst_loading", "t_res"]
audit = {
    "sources": sources,
    "summit_commit": SUMMIT_COMMIT,
    "paper": SOURCE_URL,
    "data_origin": "Reizman et al., React. Chem. Eng. 2016, 1, 658 (10.1039/C6RE00153J)",
    "rows": {"experiments": int(len(table)), "paper_reports": 387,
             "dropped_ligand": {"names": DROP_LIGANDS, "rows": int(len(dropped)),
                                "by_case": dropped.case.value_counts().sort_index().to_dict(),
                                "reason": "bidentate XantPhos is not in Kraken (monophosphines only)"},
             "kept": int(len(reactions))},
    "replicate_rows_sharing_conditions": int(reactions.duplicated(keys, keep=False).sum()),
    "case_scheme": {str(c): {"aryl_halide": ARYL_HALIDES[a][0], "aryl_halide_smiles": ARYL_HALIDES[a][1],
                             "boron_reagent": BORON[b][0], "boron_reagent_smiles": BORON[b][1],
                             "fu_scheme": f"{a}-{b}"} for c, (a, b) in CASE_SCHEME.items()},
    "units": {"temperature": "degC", "catalyst_loading": "mol%", "t_res": "s", TARGET: "%"},
}
print(f"kept {len(reactions)} of {len(table)}; dropped {len(dropped)} {DROP_LIGANDS} rows "
      f"{audit['rows']['dropped_ligand']['by_case']}")
print(f"{audit['replicate_rows_sharing_conditions']} rows share their conditions with a replicate")
display(reactions.head(3))

kept 332 of 385; dropped 53 ['XantPhos'] rows {1: 37, 2: 5, 3: 5, 4: 6}
41 rows share their conditions with a replicate


,row_id,case,catalyst,scaffold,ligand,aryl_halide,boron_reagent,temperature,catalyst_loading,t_res,yield,ton,kraken_id
0,reaction:1,1,P1-L3,P1,RuPhos,3-bromoquinoline,"3,5-dimethylisoxazole-4-boronic acid pinacol ester",30.0,0.498,600.0,0.6,1.1,4
1,reaction:2,1,P1-L6,P1,PPh3,3-bromoquinoline,"3,5-dimethylisoxazole-4-boronic acid pinacol ester",30.0,2.515,600.0,0.6,0.2,17
2,reaction:3,1,P1-L1,P1,XPhos,3-bromoquinoline,"3,5-dimethylisoxazole-4-boronic acid pinacol ester",30.0,0.513,60.0,0.6,1.1,1


## 4. The ligand mapping

Six ligands, each in the Kraken reference. Check the two SMILES agree
chemically — they come from different sources and are written differently.

In [5]:
identifiers = prep.kraken_identifiers().set_index("id")
display(pd.DataFrame([{"ligand": name, "code": next(k for k, v in LIGANDS.items() if v == name),
                       "kraken_id": kid, "kraken_name": identifiers.loc[kid, "ligand"],
                       "kraken_smiles": identifiers.loc[kid, "can_smiles"]}
                      for name, kid in MAPPING.items()]))
display(prep.describe(reactions, {"data": {"target": TARGET, "group": GROUP}}))

,ligand,code,kraken_id,kraken_name,kraken_smiles
0,XPhos,L1,1,Xphos,CC(C)c1cc(C(C)C)c(-c2ccccc2P(C2CCCCC2)C2CCCCC2)c(C(C)C)c1
1,SPhos,L2,3,SPhos,COc1cccc(OC)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1
2,RuPhos,L3,4,RuPhos,CC(C)Oc1cccc(OC(C)C)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1
3,PCy3,L5,11,PCy3,C1CCC(P(C2CCCCC2)C2CCCCC2)CC1
4,PPh3,L6,17,PPh3,c1ccc(P(c2ccccc2)c2ccccc2)cc1
5,PtBu3,L7,8,PtBu3,CC(C)(C)P(C(C)(C)C)C(C)(C)C


,ligand,kraken_id,n,mean,sd,min,max
0,XPhos,1,110,50.950909,36.053703,0.1,99.8
1,RuPhos,4,66,52.730303,34.351419,0.1,91.8
2,SPhos,3,60,60.218333,34.246838,0.1,99.5
3,PCy3,11,45,24.126667,19.957926,0.1,64.3
4,PtBu3,8,32,26.765625,29.937371,0.1,85.4
5,PPh3,17,19,5.963158,11.738267,0.1,40.4


## 5. Configure the bundle and its sections

The hub's standard sections, unchanged in meaning: `group_ohe` (ligand
one-hot), `selected_2`, the two `vbur_*_vmin` pairs, `selected_5`, `pc_top`,
`pc_scores`. Every one also carries the condition one-hots (aryl halide, boron
reagent, scaffold) and the three continuous conditions. Methods default to
`lolo`, its matched control `iid_stratified_6`, `kfold_stratified_5` and
`kfold_5`.

LOLO folds are very unequal here (XPhos holds 110 rows, PPh3 19), so read
pooled metrics rather than fold means.

In [6]:
cfg = prep.bundle_config(
    dataset=DATASET, display_name=DISPLAY_NAME, target=TARGET, group=GROUP,
    categorical=CATEGORICAL, numeric=NUMERIC, reactions=reactions, source=SOURCE_URL,
    extra={"data_origin": audit["data_origin"],
           "notes": "group = ligand; scaffold (P1 OMs / P2 Cl) is a condition; "
                    "XantPhos (L4) rows dropped; continuous conditions numeric"})
display(pd.json_normalize(cfg["data"]).T.rename(columns={0: "value"}))
display(prep.check_bundle(reactions, cfg, prep.kraken_features()))

reference = prep.reference_pca()
widths = []
for model in cfg["features"]["models"]:
    numeric, categorical = model_columns(model, cfg, reference)
    widths.append({"section": model,
                   "ligand descriptors": len([c for c in numeric if c not in NUMERIC]),
                   "continuous conditions": len([c for c in numeric if c in NUMERIC]),
                   "one-hot": int(sum(reactions[c].nunique() for c in categorical)),
                   "one-hot fields": ", ".join(categorical)})
widths = pd.DataFrame(widths)
widths["encoded"] = widths[["ligand descriptors", "continuous conditions", "one-hot"]].sum(axis=1)
display(widths)
print("methods:", cfg["evaluation"]["methods"])

,value
target,yield
group,ligand
common_categorical,"[aryl_halide, boron_reagent, scaffold]"
common_numeric,"[temperature, catalyst_loading, t_res]"
expected_rows,332
expected_groups,6


,check,ok,detail
0,row_id present and unique,True,332 rows
1,target is numeric and complete,True,yield
2,group column present,True,6 distinct
3,condition columns present,True,"aryl_halide, boron_reagent, scaffold"
4,continuous condition columns present and finite,True,"temperature, catalyst_loading, t_res"
5,kraken_id on every row,True,6 ids
6,descriptors for every kraken_id,True,all present
7,section descriptors present and finite,True,6/6 columns
8,PC1..PC4 present,True,reference projection
9,row count matches config,True,332 vs 332


,section,ligand descriptors,continuous conditions,one-hot,one-hot fields,encoded
0,group_ohe,0,3,14,"ligand, aryl_halide, boron_reagent, scaffold",17
1,selected_2,2,3,8,"aryl_halide, boron_reagent, scaffold",13
2,vbur_min_vmin,2,3,8,"aryl_halide, boron_reagent, scaffold",13
3,vbur_boltz_vmin,2,3,8,"aryl_halide, boron_reagent, scaffold",13
4,selected_5,5,3,8,"aryl_halide, boron_reagent, scaffold",16
5,pc_top,12,3,8,"aryl_halide, boron_reagent, scaffold",23
6,pc_scores,4,3,8,"aryl_halide, boron_reagent, scaffold",15


methods: ['lolo', 'iid_stratified_6', 'kfold_stratified_5', 'kfold_5']


## 6. Write the bundle, then read it back through the engine

In [7]:
bundle = prep.write_bundle(OUT, reactions, cfg, MAPPING, audit=audit)
back, ligands, reference, prepared = load_bundle(bundle)
encoded = []
for model in prepared["features"]["models"]:
    X = feature_frame(back, ligands, model, prepared, reference)
    Z = _preprocessor(X, model, prepared, reference).fit_transform(X)
    assert np.isfinite(Z).all()
    encoded.append({"section": model, "rows": Z.shape[0], "encoded inputs": Z.shape[1]})
encoded = pd.DataFrame(encoded)
assert (encoded["encoded inputs"].to_numpy() == widths["encoded"].to_numpy()).all()
display(encoded)
display(pd.DataFrame([{"file": p.name, "KB": round(p.stat().st_size / 1024, 1)}
                      for p in sorted(bundle.iterdir())]))

bundle -> C:\Users\kagoble\Documents\GitHub\DOPE-MURI\gp_collab_hub\datasets\reizman_summit\inputs


  332 rows, 6 ligands, 6 descriptor rows


,section,rows,encoded inputs
0,group_ohe,332,17
1,selected_2,332,13
2,vbur_min_vmin,332,13
3,vbur_boltz_vmin,332,13
4,selected_5,332,16
5,pc_top,332,23
6,pc_scores,332,15


,file,KB
0,audit.json,3.0
1,config.json,1.2
2,ligand_features.csv,16.7
3,ligand_mapping.csv,0.4
4,pca_reference.json,6.0
5,pca_reference.npz,12.1
6,reactions.csv,37.6


## 7. Run config and the HPC zip

GP and cluster settings are copied from the Ahneman run config, so the three
datasets' runs differ only in their data. 7 sections × 4 methods = **28 jobs**.
Run tag, zip and results folder: `reizman_summit_<date>`.

In [8]:
# ---- Editable -----------------------------------------------------------
TEMPLATE = ROOT / "configs" / "ahneman_vmin_20260924.json"
RUN_TAG  = f"{DATASET}_{date.today():%Y%m%d}"
# --------------------------------------------------------------------------

template = read_json(TEMPLATE)
run_config = {
    "schema_version": 2,
    "dataset": DATASET,
    "run_tag": RUN_TAG,
    "paths": {"bundle": f"datasets/{DATASET}/inputs", "runs": f"runs/{RUN_TAG}"},
    "run": {**template["run"], "models": list(prepared["features"]["models"]),
            "methods": list(prepared["evaluation"]["methods"]), "feature_columns": {}},
    "gp": dict(template["gp"]),
    "hpc": dict(template["hpc"]),
}
validate_config(run_config)
bundle_mod.check_methods(run_config["run"]["methods"], prepared)
check_run_against_bundle(run_config["run"]["models"], prepared, run_config["gp"])

config_path = ROOT / "configs" / f"{RUN_TAG}.json"
write_json(config_path, run_config)
try:
    write_yaml(config_path.with_suffix(".yaml"), run_config)
except ImportError:
    pass

zip_path = bundle_mod.write_zip(run_config, config_path=config_path)

import zipfile
with zipfile.ZipFile(zip_path) as archive:
    contents = pd.DataFrame([{"file": i.filename.split("/", 1)[1],
                              "KB": round(i.file_size / 1024, 1)}
                             for i in archive.infolist()])
display(contents[contents.file.str.startswith(("inputs/", "configs/"))])
print(f"{len(contents)} files, {zip_path.stat().st_size / 1024:,.0f} KB -> upload {zip_path}")

C:\Users\kagoble\Documents\GitHub\DOPE-MURI\gp_collab_hub\exports\reizman_summit_20261001.zip  (119 KB, 38 files)
  run tag   reizman_summit_20261001
  tasks     28 (7 sections x 4 methods)
  walltime  00:20:00 per task on gpu:l40:1


,file,KB
28,inputs/config.json,1.2
29,inputs/reactions.csv,37.6
30,inputs/ligand_features.csv,16.7
31,inputs/pca_reference.json,6.0
32,inputs/pca_reference.npz,12.1
33,inputs/ligand_mapping.csv,0.4
34,configs/default.json,1.3
35,configs/reizman_summit_20261001.json,1.3


38 files, 119 KB -> upload C:\Users\kagoble\Documents\GitHub\DOPE-MURI\gp_collab_hub\exports\reizman_summit_20261001.zip


## 8. Next: Fu's own descriptors

Fu's 41 QM descriptors (aryl halide, boron reagent and precatalyst; their
Table 4) plus the three conditions were their DNN's 44 inputs. Once their ESI
table is in hand, a sibling bundle `datasets/reizman_summit_fu_qm/` can carry it as
`reaction_features.csv` — the mechanism `ahneman_doyle_dft` uses — giving
"their features + our ligand sections" and "ours in place of their precatalyst
descriptors", on these same 332 rows and folds.